En este notebook se ejecuta el Análisis Exploratorio de los Datos.
Se genera un reporte sobre lo encontrado y sobre las correcciones que es necesario efectuar.

In [8]:
from _bootstrap import setup
from src.vasijas.reporting import ProcessingReport

In [9]:
PATHS = setup()
report = ProcessingReport()

In [10]:
# Ubicaciones detectadas en los datos originales

ubicaciones = [item.name for item in PATHS.DATA_ORIGINALES.iterdir() if item.is_dir()]

report.add_metric("cantidad_ubicaciones encontradas", len(ubicaciones))
report.add_metric("ubicaciones_detectadas", ubicaciones)

print(f"Cantidad de ubicaciones: {len(ubicaciones)}\n")
for ubicacion in ubicaciones:
    print(ubicacion)

Cantidad de ubicaciones: 22

Puerta Norte
Loma del Peinado
Giribaile
Puente del Obispo
Atalayuelas
Puente Tablas
Almedinilla
Baza
Cazalilla
Cerro de la Compañía
Estacar de Luciano
La Guardia
Molino de Caldona
Castellones de Ceal
Cerrillo de los Gordos
Baños de la Muela
Tútugi
Marroquies Bajos
Estacar de Robarinas
La Bobadilla
Los Patos
Toya


In [15]:
# Perfiles encontrados por ubicación

total_png = 0
conteo_png: dict[str, int] = dict()

for ruta_ubicacion in PATHS.DATA_ORIGINALES.iterdir():
    if ruta_ubicacion.is_dir():
        pngs = list(ruta_ubicacion.glob("*.png"))
        conteo_png[ruta_ubicacion.name] = len(pngs)
        total_png += len(pngs)

print(f"Total de perfiles: {total_png}\n")

for ubicacion, conteo in conteo_png.items():
    print(f"{ubicacion}: {conteo}")


report.add_metric("total_perfiles", total_png)
report.add_metric("perfiles_por_ubicacion", conteo_png)

Total de perfiles: 1242

Puerta Norte: 135
Loma del Peinado: 43
Giribaile: 7
Puente del Obispo: 28
Atalayuelas: 35
Puente Tablas: 4
Almedinilla: 66
Baza: 368
Cazalilla: 12
Cerro de la Compañía: 4
Estacar de Luciano: 16
La Guardia: 70
Molino de Caldona: 3
Castellones de Ceal: 182
Cerrillo de los Gordos: 30
Baños de la Muela: 6
Tútugi: 100
Marroquies Bajos: 18
Estacar de Robarinas: 27
La Bobadilla: 51
Los Patos: 16
Toya: 21


In [12]:
# Comprobación de concordancia entre metadata.csv y archivos de perfiles
# Correctos

In [21]:
import pandas as pd

archivos_png = {p.name for p in PATHS.DATA_ORIGINALES.rglob("*.png")}
archivo_csv = PATHS.DATA_ORIGINALES / "metadata.csv"

df = pd.read_csv(archivo_csv)

df["archivo_encontrado"] = (
    df["Identificador"]
    .astype(str)
    .str.strip()
    .add(".png")
    .isin(archivos_png)
)

no_encontrados = df.loc[~df["archivo_encontrado"], "Identificador"].astype(str).str.strip().add(".png")
print("Archivos no encontrados:")
print(no_encontrados.to_string(index=False))

cantidad_encontrados = int(df["archivo_encontrado"].sum())
cantidad_no_encontrados = int((~df["archivo_encontrado"]).sum())

report.add_metric("archivos_encontrados", cantidad_encontrados)
report.add_metric("archivos_no_encontrados", cantidad_no_encontrados)

print(f"Cantidad de encontrados: {cantidad_encontrados}\n")
print(f"Cantidad de no encontrados: {cantidad_no_encontrados}\n")


Archivos no encontrados:
      Al_03G.png
      Al_06A.png
     BA115_5.png
     BA132_1.png
      BO_3_1.png
      BO_4_1.png
      BO_4_2.png
      BO_4_3.png
      BO_5_1.png
 BO_5_1plato.png
      BO_6_1.png
      BO_6_2.png
      BO_7_1.png
      BO_7_2.png
      BO_9_1.png
      BO_9_2.png
      BO_9_3.png
      BO_9_4.png
     BO_10_1.png
BO_10_1plato.png
    BO_11a_1.png
    BO_11a_2.png
    BO_11a_3.png
    BO_11b_1.png
     BO_14_1.png
     BO_14_2.png
     BO_16_1.png
     BO_17_1.png
     BO_18_1.png
     BO_18_2.png
     BO_19_1.png
     BO_19_2.png
     BO_20_1.png
     BO_21_1.png
     BO_21_2.png
     BO_22_1.png
     BO_23_1.png
     BO_24_1.png
     BO_26_1.png
     BO_27_1.png
     BO_27_2.png
     BO_28_1.png
     BO_28_2.png
     BO_29_1.png
     BO_29_2.png
     BO_29_3.png
     BO_29_4.png
     BO_29_5.png
     BO_30_1.png
     BO_30_2.png
     BO_31_1.png
     BO_31_2.png
     BO_31_3.png
     BO_31_4.png
     BO3_2_1.png
     CC0_8_2.png
    CC_11_14.png
    CC

In [14]:
report.save_markdown(PATHS.REPORTS / "processing_report.md")